<a href="https://colab.research.google.com/github/bdh283ako/nurmosenkurssi/blob/main/Tehtava4_Verkkoteoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tehtävä 4: Verkkoteoria – Strategiset päätöksentekomallit

## Oppimistavoitteet ja lähtötiedot
Tässä tehtävässä tutustutaan verkkoteoriaan ja sen käyttöön päätöksenteossa. Tehtävässä opitaan esittämään asiat selkeästi sekä dokumentoimaan ja selittämään tehtyjä laskentapohjia peruskäyttäjälle ja kehittäjälle.

Verkkokaaviossa solmun sisällä oleva luku kertoo toiminnon keston (viikkoina) ja suluissa oleva luku ilmoittaa tarvittavien apumiesten määrän.

## Selvitettävät asiat
Tehtävässä selvitetään annetusta verkosta seuraavat asiat:
1. **Projektin lyhin suoritusaika**
2. **Varhaisimmat ja myöhäisimmät ajankohdat ($T_e, T_l$) sekä pelivarat (kokonais- ja vapaa pelivara)**
3. **Gantt-kaavio (perusmuodossaan)**
4. **Keittiökalusteiden asennuksen alkamis- ja valmistumisajankohta**
5. *(Huomautus: Tehtävänannon mukaisesti kohdan 5 apumiesten määrän optimointia ei tarvitse tehdä.)*

## Käyttö- ja kehittäjäohjeet

### Ohjeet peruskäyttäjälle
1. **Aja koodisolut järjestyksessä** valitsemalla solu ja painamalla `Shift + Enter` tai painamalla Jupyter Notebookin yläpalkista *Run All*.
2. **Tulostaulukko:** Ensimmäinen koodisolu laskee verkkoteorian mukaiset aikataulut ja tulostaa yhteenvetotaulukon:
   - **Kesto:** Toiminnon kesto viikkoina.
   - **ES ($T_e$ alku):** Varhaisin mahdollisin alkamisaika.
   - **EF ($T_e$ loppu):** Varhaisin mahdollisin valmistumisaika.
   - **LS ($T_l$ alku):** Myöhäisin sallittu alkamisaika ilman projektin viivästymistä.
   - **LF ($T_l$ loppu):** Myöhäisin sallittu valmistumisaika.
   - **TF (Total Float):** Kokonaispelivara. Jos $TF = 0$, toiminto on kriittisellä polulla.
   - **FF (Free Float):** Vapaa pelivara seuraavien toimintojen alkamiseen vaikuttamatta.
3. **Gantt-kaavio:** Toinen koodisolu visualisoi aikataulun kaaviona, jossa kriittisen polun tehtävät on korostettu punaisella värillä.

### Ohjeet kehittäjälle
- **Kirjastoriippuvuudet:** Koodi hyödyntää `networkx`-kirjastoa verkko-optimointiin ja topologiseen järjestämiseen, `pandas`-kirjastoa datan taulukointiin sekä `matplotlib`-kirjastoa visualisointiin.
- **Verkon rakenne:** Toiminnot ja niiden riippuvuudet on määritelty `tasks`-sanakirjassa (`dict`), jossa jokaisella solmulla on `duration` (kesto) ja `preds` (edeltävät tehtävät). Verkkoa voidaan muokata lisäämällä tai muuttamalla näitä arvoja.
- **Laskenta-algoritmi (CPM):**
  - *Forward Pass:* Laskee $ES$ ja $EF$ topologisessa järjestyksessä.
  - *Backward Pass:* Laskee $LS$ ja $LF$ käänteisessä topologisessa järjestyksessä.
  - *Pelivarat:* $TF = LS - ES$ ja $FF = \min(ES_{\text{seuraajat}}) - EF$.

## Kriittisen polun menetelmä (CPM)

Kriittisen polun menetelmä (Critical Path Method, CPM) on projektinhallinnan tekniikka, joka analysoi projektiin kuuluvat toiminnot, niiden kestot ja riippuvuussuhteet kriittisen polun määrittämiseksi. Kriittinen polku on toimintojen sarja, joka on suoritettava aikataulussa, jotta koko projekti valmistuu ajallaan. Mikä tahansa viivästys kriittisen polun toiminnossa vaikuttaa suoraan projektin kokonaiskestoon.

### CPM:n avainkäsitteet

CPM-menetelmän ymmärtämiseksi jokaiselle toiminnolle lasketaan useita aikaan liittyviä tunnuslukuja:

*   **Varhaisin aloitus (ES, Early Start)**: Aikaisin mahdollinen hetki, jolloin toiminto voi alkaa olettaen, että kaikki sitä edeltävät toiminnot on suoritettu.

*   **Varhaisin lopetus (EF, Early Finish)**: Aikaisin mahdollinen hetki, jolloin toiminto voi valmistua. Se lasketaan kaavalla `ES + Kesto`.

*   **Myöhäisin aloitus (LS, Late Start)**: Myöhäisin mahdollinen hetki, jolloin toiminto voi alkaa viivästyttämättä projektin kokonaisvalmistumista.

*   **Myöhäisin lopetus (LF, Late Finish)**: Myöhäisin mahdollinen hetki, jolloin toiminto voi valmistua viivästyttämättä projektin kokonaisvalmistumista. Se lasketaan kaavalla `LS + Kesto`.

*   **Kokonaispelivara (TF, Total Float)**: Kertoo, kuinka paljon toimintoa voidaan viivästyttää ilman, että koko projektin valmistuminen viivästyy. Kriittisellä polulla olevien toimintojen kokonaispelivara on nolla.
    `TF = LS - ES` tai `TF = LF - EF`.

*   **Vapaa pelivara (FF, Free Float)**: Aika, jonka toiminto voi viivästyä viivästyttämättä minkään *seuraavan* toiminnon varhaisinta aloitusta. Vapaa pelivara on aina pienempi tai yhtä suuri kuin kokonaispelivara.

In [ ]:
import pandas as pd
import networkx as nx

# 1. Määritellään verkon solmut, kestot ja edeltävyyssuhteet
#    Kukin 'task' on sanakirja, joka sisältää toiminnon 'duration' (kesto viikkoina)
#    ja 'preds' (lista edeltävistä toiminnoista).
#    'ALKU' ja 'LOPPU' ovat nollakestoisia aloitus- ja lopetussolmuja.
tasks = {
    'ALKU': {'duration': 0, 'preds': []},
    'KAIVUU': {'duration': 4, 'preds': ['ALKU']},
    'PERUSTUS': {'duration': 2, 'preds': ['KAIVUU']},
    'SADEVESIKAINOT': {'duration': 1, 'preds': ['PERUSTUS']},
    'RUNGON PYSTYTYS': {'duration': 4, 'preds': ['PERUSTUS']},
    'POHJAKERROKSEN PUTKITYÖT': {'duration': 1, 'preds': ['PERUSTUS']},
    'POHJA-KERR. LATTIA': {'duration': 2, 'preds': ['POHJAKERROKSEN PUTKITYÖT']},
    'MUURAUS': {'duration': 6, 'preds': ['RUNGON PYSTYTYS']},
    'VESIKATTO': {'duration': 2, 'preds': ['MUURAUS']},
    'RÄYSTÄÄT RÄNNIT': {'duration': 1, 'preds': ['MUURAUS']},
    'SÄHKÖTYÖT': {'duration': 2, 'preds': ['RUNGON PYSTYTYS']},
    'LÄMMITYS': {'duration': 4, 'preds': ['RUNGON PYSTYTYS']},
    'PUTKITYÖ RUNKOPUTKET': {'duration': 3, 'preds': ['POHJA-KERR. LATTIA']},
    'RAPPAUS': {'duration': 10, 'preds': ['SÄHKÖTYÖT', 'LÄMMITYS', 'PUTKITYÖ RUNKOPUTKET']},
    'PIHAN TASOITUS': {'duration': 2, 'preds': ['SADEVESIKAINOT', 'RÄYSTÄÄT RÄNNIT']},
    'KÄYTÄVÄT JA PUUTARHA': {'duration': 5, 'preds': ['PIHAN TASOITUS']},
    'LATTIOIDEN VIIMEISTELY': {'duration': 3, 'preds': ['RAPPAUS']},
    'PUUTÖIDEN VIIMEISTELY': {'duration': 3, 'preds': ['LATTIOIDEN VIIMEISTELY']},
    'KEITTIÖKALUSTEET': {'duration': 1, 'preds': ['LATTIOIDEN VIIMEISTELY']},
    'PUTKITÖIDEN VIIMEISTELY': {'duration': 2, 'preds': ['LATTIOIDEN VIIMEISTELY']},
    'MAALAUS': {'duration': 3, 'preds': ['KEITTIÖKALUSTEET', 'PUTKITÖIDEN VIIMEISTELY']},
    'LATTIOIDEN LAKKAUS': {'duration': 2, 'preds': ['PUUTÖIDEN VIIMEISTELY', 'MAALAUS']},
    'SÄHKÖTÖIDEN VIIMEISTELY': {'duration': 1, 'preds': ['MAALAUS']},
    'LOPPU': {'duration': 0, 'preds': ['VESIKATTO', 'KÄYTÄVÄT JA PUUTARHA', 'LATTIOIDEN LAKKAUS', 'SÄHKÖTÖIDEN VIIMEISTELY']}
}

# 2. Luodaan suunnattu verkko (Directed Acyclic Graph, DAG)
#    Käytetään networkx-kirjastoa verkon rakentamiseen. Solmuille lisätään duration-attribuutti.
#    Kaaret lisätään edeltävyyssuhteiden perusteella.
G = nx.DiGraph()
for task, data in tasks.items():
    G.add_node(task, duration=data['duration'])
    for pred in data['preds']:
        G.add_edge(pred, task)

#    Määritetään topologinen järjestys, joka on välttämätön eteen- ja taaksepäinlaskennassa.
topo_order = list(nx.topological_sort(G))

# 3. Eteenpäinlaskenta (Forward Pass: ES ja EF)
#    Lasketaan kunkin toiminnon Early Start (ES) ja Early Finish (EF).
#    ES: Toiminnon aikaisin mahdollinen aloitushetki.
#    EF: Toiminnon aikaisin mahdollinen valmistumishetki (ES + Kesto).
ES = {node: 0 for node in G.nodes()}
EF = {node: 0 for node in G.nodes()}

for node in topo_order:
    dur = G.nodes[node]['duration']
    preds = list(G.predecessors(node))
    if preds:
        # ES on edeltävien toimintojen maksimi EF.
        ES[node] = max(EF[p] for p in preds)
    else:
        # Aloitustoiminnoilla (ei edeltäjiä) ES on 0.
        ES[node] = 0
    EF[node] = ES[node] + dur

# Projektin kokonaiskesto on LOPPU-solmun EF.
project_duration = EF['LOPPU']

# 4. Taaksepäinlaskenta (Backward Pass: LS ja LF)
#    Lasketaan kunkin toiminnon Late Start (LS) ja Late Finish (LF).
#    LF: Myöhäisin mahdollinen valmistumishetki ilman projektin viivästyttämistä.
#    LS: Myöhäisin mahdollinen aloitushetki (LF - Kesto).
LF = {node: project_duration for node in G.nodes()}
LS = {node: project_duration for node in G.nodes()}

# Käydään läpi solmut käännetyssä topologisessa järjestyksessä.
for node in reversed(topo_order):
    dur = G.nodes[node]['duration']
    succs = list(G.successors(node))
    if succs:
        # LF on seuraavien toimintojen minimi LS.
        LF[node] = min(LS[s] for s in succs)
    else:
        # Lopetustoiminnoilla (ei seuraajia) LF on projektin kesto.
        LF[node] = project_duration
    LS[node] = LF[node] - dur

# 5. Pelivarojen laskenta (TF ja FF)
#    TF (Total Float): Kokonaispelivara = LS - ES tai LF - EF. Jos TF=0, toiminto on kriittisellä polulla.
#    FF (Free Float): Vapaa pelivara = Seuraavien toimintojen minimi ES - Toiminnon EF.
TF = {node: LS[node] - ES[node] for node in G.nodes()}
FF = {}
for node in G.nodes():
    succs = list(G.successors(node))
    if succs:
        # FF on lyhin aika, jolla tämä toiminto voi viivästyä vaikuttamatta seuraavan toiminnon ES:ään.
        FF[node] = min(ES[s] for s in succs) - EF[node]
    else:
        # Lopetustoiminnoilla FF on projektin kesto - toiminnon EF.
        FF[node] = EF['LOPPU'] - EF[node]

# 6. Tulostaulukko
#    Kootaan lasketut arvot ja kriittinen polku DataFrameen ja tulostetaan.
df = pd.DataFrame({
    'Kesto (vko)': [tasks[n]['duration'] for n in topo_order],
    'ES (Te_alku)': [ES[n] for n in topo_order],
    'EF (Te_loppu)': [EF[n] for n in topo_order],
    'LS (Tl_alku)': [LS[n] for n in topo_order],
    'LF (Tl_loppu)': [LF[n] for n in topo_order],
    'Kokonaispelivara (TF)': [TF[n] for n in topo_order],
    'Vapaa pelivara (FF)': [FF[n] for n in topo_order],
    'Kriittinen': [TF[n] == 0 for n in topo_order] # Kriittiset toiminnot, jos TF on 0
}, index=topo_order)

# Tulostetaan projektin lyhin suoritusaika ja koko tulostaulukko.
print(f"Projektin lyhin suoritusaika: {project_duration} viikkoa\n")
display(df)


In [ ]:
import matplotlib.pyplot as plt

# Visualisoidaan toiminnalliset tehtävät (kesto > 0)
fig, ax = plt.subplots(figsize=(12, 8))
nodes_to_plot = [n for n in reversed(topo_order) if tasks[n]['duration'] > 0]

for i, node in enumerate(nodes_to_plot):
    is_crit = TF[node] == 0
    color = '#d9534f' if is_crit else '#5bc0de'
    ax.barh(node, tasks[node]['duration'], left=ES[node], color=color, edgecolor='black', alpha=0.85)
    ax.text(ES[node] + 0.1, i, f"ES:{ES[node]} / LF:{LF[node]} (TF:{TF[node]})",
            va='center', ha='left', fontsize=8, color='black', fontweight='bold')

ax.set_xlabel('Projektin aika (viikkoina)', fontsize=12)
ax.set_title('Gantt-kaavio – Tehtävä 4: Verkkoteoria', fontsize=14, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Yhteenveto tuloksista

1. Projektin **lyhin suoritusaika** on **34 viikkoa**.
2. Te, Tl ja pelivarat löytyy suoraan koodisolun NF2PyweYyknl tulostamasta taulukosta ennen Ganttia.
3. Gantt kaavio yläpuolella.
4. Keittiökalusteiden asennus:
   - **Alkamisajankohta ($ES$):** Voi alkaa varhaimmillaan viikon **27** lopussa (viikolla **28**).
   - **Valmistumisajankohta ($LF$):** Pitää olla valmis viimeistään viikolla **29**.
   - **Pelivara ($TF, FF$):** Kokonais- ja vapaa pelivara ovat **1 viikko**.
